In [1]:
import os
import sys
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(ROOT)
from src.analysis.queries import get_volume_com_dimensoes

import pandas as pd
import numpy as np 

In [2]:
df = get_volume_com_dimensoes()

c:\Users\bruno\OneDrive\Desktop\clamed v4\projeto-etl-iqvia-clamed\src\analysis\queries.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=params)


In [3]:
df.head()
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   nome_regiao    19 non-null     object 
 1   nome_bandeira  19 non-null     object 
 2   cod_ean        19 non-null     int64  
 3   volume_venda   19 non-null     float64
 4   periodo        19 non-null     object 
dtypes: float64(1), int64(1), object(3)
memory usage: 892.0+ bytes


,cod_ean,volume_venda
count,1.900000e+01,19.000000
mean,4.180598e+07,22.315789
std,2.209514e+06,29.955620
min,3.268915e+07,1.000000
25%,4.227722e+07,4.000000
50%,4.235546e+07,7.000000
75%,4.236041e+07,32.000000
max,4.238925e+07,109.000000


Volume total vendido

In [4]:
df["volume_venda"].sum()

np.float64(424.0)

Volume por bandeira

In [5]:
df.groupby("nome_bandeira", as_index=False).agg(
    volume_total=("volume_venda", "sum"),
    media=("volume_venda", "mean"),
    maximo=("volume_venda", "max")
)


,nome_bandeira,volume_total,media,maximo
0,Bandeiras Concorrente SI,51.0,12.750,39.0
1,Bandeiras Concorrentes SO,275.0,34.375,109.0
2,Preco Popular,98.0,14.000,42.0


Volume por regiao (tem só uma por enquanto)

In [6]:
df.groupby("nome_regiao")["volume_venda"].sum()

nome_regiao
CAMPO GRANDE - CENTRO    424.0
Name: volume_venda, dtype: float64

Produtos mais vendidos

In [7]:
df.groupby("cod_ean", as_index=False).agg(
    volume_total=("volume_venda", "sum")
).sort_values("volume_total", ascending=False)


,cod_ean,volume_total
7,42360414,190.0
3,42277217,115.0
6,42360407,56.0
8,42389248,36.0
1,42110200,12.0
0,32689150,5.0
2,42176763,4.0
4,42355014,3.0
5,42355465,3.0


Comparar bandeira por produto

In [8]:
df.pivot_table(
    index="cod_ean",
    columns="nome_bandeira",
    values="volume_venda",
    aggfunc="sum",
    fill_value=0
)


nome_bandeira,Bandeiras Concorrente SI,Bandeiras Concorrentes SO,Preco Popular
cod_ean,,,
32689150,0.0,0.0,5.0
42110200,0.0,7.0,5.0
42176763,0.0,4.0,0.0
42277217,7.0,85.0,23.0
42355014,0.0,2.0,1.0
42355465,0.0,3.0,0.0
42360407,1.0,40.0,15.0
42360414,39.0,109.0,42.0
42389248,4.0,25.0,7.0
